# Cluster-GCN on Reddit Graph

Node Classification on Reddit: Training large-scale GCN on Reddit by partitioning nodes into subgraphs. This notebook implements the approach with `SAGEConv` inside a `K3RedditNet` model, trained with the Adam optimizer for 10 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `SAGEConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "torch")

import keras
from keras import layers, ops

import k3_node
from k3_node import datasets, layers as k3_layers
from k3_node.loader import ClusterData, ClusterLoader

title = "Cluster-GCN Graph Partitioning with SAGEConv"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & Cluster-GCN Partitioning (safe for memory)
dataset = datasets.Planetoid(root="./data/Planetoid", name="PubMed")
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

cluster_data = ClusterData(data, num_parts=20)
train_loader = ClusterLoader(cluster_data, batch_size=4, shuffle=True)

# 2. SAGEConv Model
class K3ClusterNet(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.SAGEConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.SAGEConv(hidden_channels, out_channels)
        self.dropout = layers.Dropout(0.5)

    def call(self, inputs, training=False):
        x, edge_index = inputs
        x = self.dropout(x, training=training)
        x = ops.relu(self.conv1(x, edge_index))
        x = self.dropout(x, training=training)
        return self.conv2(x, edge_index)

k3_model = K3ClusterNet(num_features, 128, num_classes)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=["acc"],
)

# 4. Multi-epoch Cluster Generator
def cluster_generator(loader):
    while True:
        for batch in loader:
            x = ops.convert_to_tensor(batch.x, dtype="float32")
            edge_index = ops.convert_to_tensor(batch.edge_index, dtype="int64")
            y = ops.convert_to_tensor(batch.y, dtype="int64")
            mask = ops.cast(batch.train_mask, "float32") if hasattr(batch, "train_mask") and batch.train_mask is not None else None
            if mask is not None and ops.convert_to_numpy(ops.sum(mask)) > 0:
                yield (x, edge_index), y, mask
            else:
                yield (x, edge_index), y

print(f"Training K3-Node Cluster-GCN on {backend} backend...")
history = k3_model.fit(
    cluster_generator(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=5,
    verbose=1,
)

# 5. Full-graph Evaluation
test_acc = data.accuracy(k3_model(data.inputs), mask="test_mask")
print(f"Test Accuracy: {test_acc:.4f}")

print("\n✓ K3-Node Cluster-GCN execution completed successfully!")
